# 오디오 스펙트로그램 + ResNet — 원문제 모범답안 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 GAITE **Synthetic Speech Detector** 모범답안과 동일): 오디오 분류를 **비전 문제**로 —
*미리 변환된 2D 오디오 특징(스펙트로그램)* 을 **1채널 ResNet** 으로 분류한다. **이 노트북 코드는 원문제 *모범답안
(Ref Result)* 골격을 그대로** 따른다(암기 매핑 1:1):

| 원문제 모범답안 | 이 노트북 |
|---|---|
| `SpectrogramDataset(Dataset)` → `{"spectrogram","label"}` | 동일 (turkey VGGish → 같은 dict) |
| `AudioNet` = **ResNet(1채널 conv1) → fc(2)** | 동일 (resnet18) |
| `train_one_epoch(model, train_loader, val_loader, ...)` | 동일 (그대로) |
| `predict(model, loader, device)` (argmax) | 동일 (그대로) |
| `save_submission_csv` | 동일 흐름 (대회 형식 `vid_id,is_turkey`) |
| Adam(lr=1e-4, wd=1e-5) · CrossEntropyLoss · random_split | 동일 |

**대회 데이터**: [Don't Call Me Turkey!](https://www.kaggle.com/c/dont-call-me-turkey) — 칠면조 울음 이진검출. 데이터가
**미리 추출된 2D 오디오 특징**(VGGish, 10프레임×128)으로 제공돼 ~11MB 로 가볍고, 합성음성 문제의 *미리 변환된 멜 .pt*
와 같은 구도(둘 다 precomputed 2D 특징 + CNN). 그 특징맵을 `(1,10,128)` 1채널 이미지로 ResNet 에 넣는다.

> ⚙️ GPU 권장(작은 ResNet, T4 ~1분).  ⚠️ API 토큰 평문 — 외부 공유 금지.
> 🔑 **첫 실행 시 Rules 수락**: 403 이면 [대회 페이지](https://www.kaggle.com/c/dont-call-me-turkey/rules)에서 'I Understand and Accept' 1회.
> 💡 이 노트북은 `weights=None`(무작위 초기화)로 학습한다 — 사전학습 다운로드 없이 가볍게(IOAI 시험환경도 인터넷이 없어 `weights=None`).

## 0. 설치 · 자격증명

In [1]:
import sys, subprocess
for pkg in ["kaggle", "torch", "torchvision", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("준비 완료")

준비 완료


## 1. Kaggle 에서 데이터 다운로드 (train.json / test.json)
403 이면 대회 Rules 를 한 번 수락(위 안내).

In [2]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("dont-call-me-turkey", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료")
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:160])
    print("→ 403 이면 https://www.kaggle.com/c/dont-call-me-turkey/rules 에서 'I Understand and Accept' 1회 후 재실행")

100%|██████████| 3.88M/3.88M [00:00<00:00, 69.3MB/s]

다운로드 완료


## 2. 데이터 준비 · 1채널 ResNet 학습 & 제출 파일 생성
`DS`: `train.json`/`test.json` 의 VGGish `audio_embedding` 을 `(10,128)` 로 **0패딩·`/255` 정규화** 해 `(1,10,128)` **1채널 텐서**로 만든다(train 은 `is_turkey` 라벨). **`resnet18(weights=None)`** 의 `conv1` 을 **1채널 `Conv2d`** 로 교체(3채널 가중치를 채널평균으로 초기화)하고 `fc` 를 **2클래스**로 → **Adam(1e-4)** 로 **5에폭** `CrossEntropyLoss` 학습 → test **argmax** 예측 → `sub.csv`(`vid_id, is_turkey`) 저장.

In [3]:
##########################
# Your code here
##########################

## 3. 제출 파일 내려받기

In [5]:
try:
    from google.colab import files; files.download("sub.csv")
except Exception:
    print("submission.csv 위치:", "sub.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🏁 제출하기
`submission.csv` 를 [Don't Call Me Turkey!](https://www.kaggle.com/c/dont-call-me-turkey/submit) 에 제출.

원문제 **모범답안** 골격(`SpectrogramDataset`→**`AudioNet`(ResNet 1채널)**→`train_one_epoch`→`predict`→제출)을 **그대로**
옮겨 *오디오를 이미지로 분류*하는 합성음성 탐지 기법을 연습했다. 핵심은 *미리 추출된 2D 오디오 특징을 1채널 ResNet 에
넣는 것*. 더: resnet34, 에폭↑, (AUC 대회면 argmax 대신 softmax 확률 제출로 점수↑), SpecAugment.